# Notebook 06 — Advanced RAG & Evaluation

**Week 8 | `rag_lib`, `llm_inspector`**

By the end of this notebook you will be able to:
- Implement hybrid BM25 + dense retrieval
- Apply cross-encoder reranking to improve result ordering
- Use HyDE (Hypothetical Document Embeddings) for query expansion
- Evaluate retrieval quality using RAGAS metrics
- Diagnose retrieval failures with `llm_inspector`

## Why Basic RAG Fails

Notebook 05 showed you how to build a working RAG pipeline. This notebook addresses the most common ways it breaks in production.

The production chunking article identified three diagnostic metrics from RAGAS that tell you *where* the failure is:

| Metric | Low score means |
|---|---|
| Context Recall | Retrieval is missing relevant chunks |
| Context Precision | Retrieval is returning irrelevant chunks |
| Faithfulness | Generation is hallucinating — not grounded in retrieved context |

Each metric points to a different fix.

## Hybrid Retrieval (BM25 + Dense)

Dense vector search excels at semantic similarity. BM25 excels at keyword matching. Neither dominates the other across all query types. Combining them consistently outperforms either alone.

The `rag_lib` retriever supports hybrid mode:

In [ ]:
from rag_lib.retrieval.retriever import Retriever

# TODO: Initialize with your ChromaDB collection from notebook 05
# retriever = Retriever(collection=collection, mode="hybrid", bm25_weight=0.3)
# results = retriever.retrieve("your query", top_k=10)
print("Exercise: compare hybrid vs dense-only results on 5 test queries."
      "\nMeasure Context Recall for each.")

## Cross-Encoder Reranking

Initial retrieval returns candidates ranked by approximate similarity. A cross-encoder reranker scores each (query, chunk) pair jointly — more accurate but slower, so applied only to the top-k candidates.

In [ ]:
from rag_lib.retrieval.reranker import Reranker

# TODO: Apply reranker to retrieval results
# reranker = Reranker(model="cross-encoder/ms-marco-MiniLM-L-6-v2")
# reranked = reranker.rerank(query, results, top_n=5)
print("Exercise: compare Context Precision before and after reranking.")

## HyDE — Hypothetical Document Embeddings

Instead of embedding the user's query directly, ask the LLM to generate a *hypothetical* answer, then embed that. The hypothesis is often closer in embedding space to real answers than the question itself.

In [ ]:
from rag_lib.retrieval.expander import QueryExpander

# TODO: Apply HyDE expansion
# expander = QueryExpander(engine=engine)
# expanded_query = expander.hyde("What is the refund policy?")
# results = retriever.retrieve(expanded_query, top_k=5)
print("Exercise: compare retrieval quality with and without HyDE "
      "on 3 questions where the question wording differs from the document wording.")

## RAGAS Evaluation

RAGAS provides automated evaluation of RAG pipelines without hand-labelled data. It uses an LLM as judge to score each metric.

In [ ]:
from rag_lib.eval.ragas_runner import RagasRunner

# TODO: Run RAGAS evaluation
# runner = RagasRunner(engine=engine)
# scores = runner.evaluate(
#     questions=[...],
#     ground_truths=[...],
#     pipeline=pipeline
# )
# print(scores)  # context_recall, context_precision, faithfulness, answer_relevancy
print("Exercise: identify which metric is lowest for your pipeline "
      "and apply the corresponding fix from the table above.")

## Diagnosing with llm_inspector

The `llm_inspector` RAG adapter makes retrieval decisions visible: what was retrieved, what was reranked, what was dropped, and why.

In [ ]:
from llm_inspector.rag import RAGInspector

# TODO: Wrap your pipeline with RAGInspector
# inspector = RAGInspector()
# inspector.add_pipeline("baseline", baseline_pipeline)
# inspector.add_pipeline("hybrid+rerank", improved_pipeline)
# results = inspector.query_all("your test query")
# inspector.print_comparison(results)
print("Exercise: run the same query through baseline and improved pipelines. "
      "\nUse llm_inspector to show which chunks were retrieved by each.")

## Exercises

1. Implement hybrid retrieval and measure Context Recall improvement.
2. Apply cross-encoder reranking and measure Context Precision improvement.
3. Apply HyDE on 3 queries where keyword mismatch is the likely failure mode.
4. Run RAGAS on your pipeline. Identify the lowest-scoring metric and fix it.
5. Use `llm_inspector` to compare baseline vs improved pipeline side-by-side.

---
**Next:** [Notebook 07 — Reference App Walkthrough](07_reference_app_walkthrough.ipynb)